# Deliverable 2

This notebook demonstrates the functionalities built as part of deliverable 2.

- From SessionIDs to SavedScenarioIDs: creating a SavedScenario in MyETM associated with a particular SessionID. 
- From SavedScenarioIDs to SessionIDs: linking SavedScenarioIDs to Session IDs
- Copy scenarios (based on SessionID)
- Read scenarios (based on SessionID)

First set up the notebook.

In [ ]:
from example_helpers import setup_notebook
from pyetm.models.scenario_packer import ScenarioPacker

setup_notebook()
packer = ScenarioPacker()

## Copy

There are three types of 'copy' implemented as part of this subdeliverable:
1. Copy from Excel
2. Simple copy
3. Deep copy

### Copy from Excel

In excel you can perform either a simple copy or deep copy. If you specify a `parent` scenario, a **simple copy** will be performed. That means the scenario is copied completely, and the 'template' scenario reference is kept (as you can see in the metadata section). A **deep copy** breaks the link to the template scenario, so the scenario is copied exactly but the template is not set.

In [ ]:
from pyetm.models.scenarios import Scenarios

scenarios = Scenarios.from_excel("../examples/inputs/example_input_excel.xlsx")

In [ ]:
# Metadata
for scenario in scenarios:
    print(f"Title: {scenario.title}")
    print(f"ID: {scenario.id}")
    print(f"Template: {scenario.template}")
    print("")

In [ ]:
# Inputs
for scenario in scenarios:
    inputs = scenario.inputs.to_dataframe(columns=["user"]).head(15)
    print(inputs)
    print("")

### Copy directly in the workbook

There are also convenience methods to copy and deep copy. You can set copy_roles to True or False (although this functionality is not fully built yet as roles have not yet been implemented).

Again simple and deep copy are both available.

In [ ]:
from pyetm.models.scenario import Scenario

original = Scenario.load(2729861)
copy = original.copy(title="New Title")
with_roles = original.copy(title="Copy with roles", copy_roles=True)

print(f"Original ID: {original.id}")
print(f"Copy ID: {copy.id}")
print(f"Copy Title: {copy.title}")
# print(f"With roles, roles: {with_roles.roles}") # Roles are not on scenarios in pyetm yet
print(f"With roles ID: {with_roles.id}")

In [ ]:
deep_copy = original.deep_copy(title="Deep copy")

print(f"Copy ID: {deep_copy.id}")

packer.add(deep_copy)
# Inputs still the same
packer.inputs(columns=["user", "default"]).head(15)

In [ ]:
print(f"Original Template:      {original.template}")
print(f"Original ID:            {original.id}")
print(f"Copy Template:          {copy.template}")
print(f"Deep Copy Template:     {deep_copy.template}")

## Read

As part of the Read deliverable, we wanted to:
1. Read updates (for example from excel) without necessarily sending them to the API
2. Only selectively send updates to the API based on what changed or your workflow

In [ ]:
# Standard mode: Load and push all data to ETM (creates/updates scenarios)
# scenarios = Scenarios.from_excel("../examples/inputs/example_input_excel.xlsx")

# Read-only mode: Load data locally but don't push anything to ETM
scenarios = Scenarios.from_excel("../examples/inputs/example_input_excel.xlsx", read_only=True)

# Selective read-only: Push user_values but skip curves and sortables
# scenarios = Scenarios.from_excel("../examples/inputs/example_input_excel.xlsx", read_only=['custom_curves', 'sortables'])

#### Read-Only Parameters

| Mode | Syntax | user_values | custom_curves | sortables |
|------|--------|-------------|---------------|-----------|
| **Standard** | `read_only=False` (default) | Upload | Upload | Upload |
| **Full Read-Only** | `read_only=True` | Skip | Skip | Skip |
| **Selective** | `read_only=['custom_curves']` | Upload | Skip | Upload |
| **Selective** | `read_only=['custom_curves', 'sortables']` | Upload | Skip | Skip |

**Note:** In all modes, data is loaded into local scenario objects. The `read_only` parameter controls whether API upload calls are made.

## SessionIDs to SavedScenarioIDs

For this subdeliverable, we introduced the Saved Scenario model, and convenience methods to save your scenario to a SavedScenario.

## SavedScenarioIDs to SessionIDs

For this subdeliverable, we added convenience methods to the Saved Scenario model to get the underlying scenario from a Saved Scenario, or to build a Saved Scenario based on a session id.

As part of these two subdeliverables, we also added methods to fetch, update and create saved scenarios, which are also demonstrated below.

### Scenario.save()

The simplest way to make a saved scenario

In [ ]:
from pyetm.models.saved_scenario import SavedScenario

# Create a session scenario
scenario = Scenario.new(area_code="nl", end_year=2050, title="My Test Scenario")
print(f"Created session scenario {scenario.id}")

# Save it to MyETM - automatically uses scenario.id, scenario.title, and scenario.private
saved_scenario = scenario.save(description="Saved using save()")

print(f"Saved as SavedScenario {saved_scenario.id}")
print(f"Title: {saved_scenario.title}")
print(f"Description: {saved_scenario.description}")

The underlying scenario's title will be taken for the saved scenario, unless another title is specified. If there are no titles, an error will be thrown.

In [ ]:
scenario2 = Scenario.new(area_code="nl", end_year=2050, title="Original Title")
saved_scenario2 = scenario2.save(title="Custom Title", private=True)

print(f"Scenario title: {scenario2.title}")
print(f"SavedScenario title: {saved_scenario2.title}")
print(f"Private: {saved_scenario2.private}")

### SavedScenario.get_scenario()

Get the underlying scenario from a saved scenario.

In [ ]:
from pyetm.clients.base_client import BaseClient
client = BaseClient()

underlying_scenario = saved_scenario.get_scenario(client)
print(f"Scenario ID: {underlying_scenario.id}")
print(f"Area code: {underlying_scenario.area_code}")
print(f"End year: {underlying_scenario.end_year}")

### SavedScenario.from_scenario()

In [ ]:
scenario3 = Scenario.new(area_code="nl", end_year=2050)
saved3 = SavedScenario.from_scenario(client, scenario3, "From scenario method")
print(f"Method 2 - SavedScenario ID: {saved3.id}")

### SavedScenario.create()

In [ ]:
saved4 = SavedScenario.create(
    client,
    params={
        "scenario_id": scenario3.id,
        "title": "Created with explicit params",
        "description": "Using SavedScenario.create()",
        "private": False
    }
)
print(f"Method 3 - SavedScenario ID: {saved4.id}")

### SavedScenario.update()

In [ ]:
saved_scenario.update(
    client,
    title="Updated Title",
    description="Updated description"
)

print(f"Updated title: {saved_scenario.title}")
print(f"Updated description: {saved_scenario.description}")